## Z-Ordering in Delta Tables

**Z-ordering** is a technique that co-locates related information (related column values) in the same set of data files within a Delta table.

- **What it does:** Reorganizes data in the files so that rows with similar values for the chosen column(s) are stored together.
- **Why use it:** If you frequently filter on a column (e.g., `customer_id`), applying Z-order on that column lets Databricks leverage **file statistics** (min/max values per file) to **skip files that don't contain the required data** — a technique called **data skipping**.
- **Benefit:** Potentially reduces the amount of data scanned, speeding up queries.

### How to apply

Use the `OPTIMIZE` command together with `ZORDER BY` on the column(s) you most frequently filter on:

```sql
OPTIMIZE catalog.schema.table_name
ZORDER BY (customer_id);
```

You can also Z-order on multiple columns:

```sql
OPTIMIZE catalog.schema.table_name
ZORDER BY (customer_id, order_date);
```

In [0]:
# Z-Ordering explained with a simple example

# 1) Create a small Delta table with customer orders
spark.sql("""
  CREATE OR REPLACE TABLE workspace.default.zorder_orders (
    order_id   INT,
    customer_id INT,
    amount     DOUBLE
  )
""")

# 2) Insert 1M rows spread across 1,000 customers
#    (data arrives in random customer order, so each file contains many customers)
spark.sql("""
  INSERT INTO workspace.default.zorder_orders
  SELECT
    id AS order_id,
    CAST(rand() * 1000 AS INT) AS customer_id,
    round(rand() * 500, 2) AS amount
  FROM range(1000000)
""")

# 3) Before Z-ordering: filter for one customer and check how many files were read
print("--- BEFORE Z-ordering ---")
spark.sql("SELECT count(*) FROM workspace.default.zorder_orders WHERE customer_id = 42").show()

# 4) Apply Z-ordering on the column we filter on (compacts files + co-locates customer_id)
spark.sql("OPTIMIZE workspace.default.zorder_orders ZORDER BY (customer_id)")

# 5) After Z-ordering: the same query can SKIP files whose min/max customer_id
#    doesn't include 42, so far fewer files are scanned
print("--- AFTER Z-ordering ---")
spark.sql("SELECT count(*) FROM workspace.default.zorder_orders WHERE customer_id = 42").show()

# Compare the 'files read' metric of both queries in Spark UI / Query Profile
# to see the data-skipping benefit of Z-ordering.

# INTERVIEW ANSWER: What is Z-ORDER BY in Delta Lake?

> **"ZORDER BY** is an option of the `OPTIMIZE` command in Delta Lake that **co-locates rows with similar values** in the columns you specify into the **same data files**. The data isn't sorted globally — instead, values are **clustered** so that each file's **min/max statistics** for those columns span a narrower range."

---

## Why it matters

Delta stores **per-file statistics** (min/max per column). When I run a query with a filter like:

```sql
SELECT * FROM orders WHERE customer_id = 42;
```

the query engine uses those statistics to **skip** any file whose min/max range for `customer_id` doesn't include 42 — this is called **data skipping**.

With Z-ordering on `customer_id`, files contain far fewer distinct customer IDs, so **many more files can be skipped** and the query scans far less data.

---

## How to use it

```sql
OPTIMIZE catalog.schema.table ZORDER BY (customer_id);

-- or multiple columns:
OPTIMIZE catalog.schema.table ZORDER BY (customer_id, order_date);
```

---

## Key points I'd add

| # | Point |
|---|-------|
| 1 | Z-ordering only happens when `OPTIMIZE` runs — it's a **maintenance operation**, not something applied at write time. |
| 2 | It also **compacts small files** into larger ones (usually ~1 GB), which further improves scan efficiency. |
| 3 | Choose columns you **actually filter on frequently** — typically high-cardinality columns used in `WHERE` clauses. |
| 4 | Z-ordering multiple columns gives **diminishing returns** — the effect gets diluted across columns; **1–4 columns** is the usual guideline. |
| 5 | It's **not a physical sort** (no global ordering guarantee) — queries still need filters / `WHERE` predicates to benefit; it won't speed up full scans. |
| 6 | **Modern alternative: Liquid Clustering** (`CLUSTER BY`) — the successor to Hive-style partitioning and Z-ordering on Databricks. It's incremental, works on writes, and doesn't require re-optimizing partitions. |
| 7 | Best with **incremental `OPTIMIZE`** — re-run it as new data arrives so the new files also get co-located. |
#

In [0]:
# PRACTICAL EXAMPLE: seeing the data-skipping benefit of ZORDER BY
# (matches the interview answer: co-locating customer_id values -> narrower
# per-file min/max stats -> files can be skipped -> less data scanned)

import time

# 1) Fresh table with 2M rows across 1,000 customers
#    (writes it in RANDOM customer order, like real-world ingested data)
spark.sql("""
  CREATE OR REPLACE TABLE workspace.default.interview_zorder_orders AS
  SELECT
    id                                       AS order_id,
    CAST(rand() * 1000 AS INT)               AS customer_id,
    date_add(current_date(), -CAST(rand() * 365 AS INT)) AS order_date,
    round(rand() * 500, 2)                   AS amount
  FROM range(2000000)
""")

# 2) Helper: how many files does the table currently have?
def num_files():
    return spark.sql("DESCRIBE DETAIL workspace.default.interview_zorder_orders") \
                .select("numFiles").collect()[0][0]

print(f"Files BEFORE optimize: {num_files()}")

# 3) Time the same filtered query BEFORE Z-ordering
t0 = time.perf_counter()
df_before = spark.sql("SELECT count(*), sum(amount) FROM workspace.default.interview_zorder_orders WHERE customer_id = 42")
df_before.collect()
before_secs = time.perf_counter() - t0
print(f"Query time BEFORE Z-ordering: {before_secs:.2f}s")

# 4) Apply OPTIMIZE ... ZORDER BY (the interview answer's core command)
spark.sql("OPTIMIZE workspace.default.interview_zorder_orders ZORDER BY (customer_id)")

print(f"Files AFTER optimize: {num_files()}  (small files compacted too)")

# 5) Same query AFTER Z-ordering -- data skipping can now prune whole files
t1 = time.perf_counter()
df_after = spark.sql("SELECT count(*), sum(amount) FROM workspace.default.interview_zorder_orders WHERE customer_id = 42")
df_after.collect()
after_secs = time.perf_counter() - t1
print(f"Query time AFTER  Z-ordering: {after_secs:.2f}s")

# 6) Prove the data skipping: per-file min/max stats for customer_id
#    After Z-ordering, each file covers a much narrower customer range,
#    so a file whose [min, max] doesn't bracket 42 is skipped entirely.
files_df = spark.read.format("delta") \
    .load("workspace.default.interview_zorder_orders") \
    .select("customer_id")

print("""
To see the actual 'files read' metric for both queries, open the
Query Profile (or Spark UI) for cells 3) and 5) above — the Z-ordered
query reads far fewer files thanks to data skipping.
""")

# 7) Modern alternative worth mentioning in an interview: Liquid Clustering
# spark.sql("""
#   CREATE OR REPLACE TABLE ... CLUSTER BY (customer_id) ...
# """)
# -- clustering happens on write + incremental OPTIMIZE, no manual ZORDER BY